# 01 - Pruebas del Motor de Recomendación (Matriz 3x3)

Este notebook tiene como objetivo validar matemáticamente las funciones de nuestro sistema de recomendación (similitud coseno y predicción de scores) antes de integrarlas a la interfaz gráfica de Streamlit.

**Escenario de Prueba:**
* **Ana:** Amante de Sci-Fi, odia la comedia. No ha visto "Duro de Matar".
* **Beto:** Le gusta Sci-Fi y la Acción.
* **Carlos:** Amante de la Comedia y la Acción, odia Sci-Fi.

En nuestro DataFrame de Pandas, utilizaremos `NaN` para representar las películas que un usuario aún no ha valorado, imitando el comportamiento del dataset real de MovieLens.

In [7]:
import sys
import numpy as np
import pandas as pd

from src.similarity import calcular_matriz_similitud
from src.recommender import recomendar

## 1. Construcción de la Matriz de Prueba
Crearemos el DataFrame tal cual lo estructuraría la función `construir_matriz` de nuestro `data_loader.py`.

In [8]:
usuarios = ['Ana', 'Beto', 'Carlos']
peliculas = ['Matrix', 'Son como niños', 'Duro de Matar']

# Calificaciones del 1 al 5. Usamos np.nan para la película que Ana no ha visto.
datos = [
    [5.0, 1.0, np.nan], # Ana
    [4.0, 1.0, 4.0],    # Beto
    [1.0, 5.0, 4.0]     # Carlos
]

matriz_prueba = pd.DataFrame(datos, index=usuarios, columns=peliculas)

print("--- Matriz de Preferencias Original (con NaNs) ---")
display(matriz_prueba)

--- Matriz de Preferencias Original (con NaNs) ---


,Matrix,Son como niños,Duro de Matar
Ana,5.0,1.0,NaN
Beto,4.0,1.0,4.0
Carlos,1.0,5.0,4.0


## 2. Validación de la Similitud Coseno
Nuestra función `calcular_matriz_similitud` está programada para rellenar temporalmente los `NaN` con ceros para poder calcular el producto punto y las normas correctamente. Según nuestros datos, Ana y Beto deberían tener la similitud más alta.

In [9]:
matriz_similitud = calcular_matriz_similitud(matriz_prueba)

print("--- Matriz de Similitud Coseno (NxN) ---")
display(matriz_similitud.round(4))

--- Matriz de Similitud Coseno (NxN) ---


,Ana,Beto,Carlos
Ana,1.0000,0.7169,0.3026
Beto,0.7169,1.0000,0.6715
Carlos,0.3026,0.6715,1.0000


## 3. Generación de Recomendación y Predicción
Llamaremos a la función principal `recomendar`. Seleccionaremos a **Ana** y buscaremos a sus vecinos. El sistema debería detectar que la película que Ana no ha visto (Duro de Matar) tiene un puntaje alto por parte de su vecino más cercano (Beto), y por lo tanto, recomendársela con un score predictivo matemático.

In [10]:
# Solicitamos recomendar 1 película basándonos en los 2 vecinos posibles
recomendacion_ana = recomendar(
    user_id='Ana', 
    matriz=matriz_prueba, 
    similitud=matriz_similitud, 
    k_vecinos=2, 
    n_recomendaciones=1
)

print("--- Recomendación Final para Ana ---")
print("Película sugerida y su score predictivo:")
print(recomendacion_ana)

--- Recomendación Final para Ana ---
Película sugerida y su score predictivo:
Duro de Matar    4.0
dtype: float64
